In [4]:
!apt update
!apt install -y mpich

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,985 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [69.9 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,066 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:14 ht

In [14]:
%%writefile mpi_program.c
#include <mpi.h>
#include <stdio.h>

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    printf("Hello from process %d out of %d\n", rank, size);

    int number;
    if (size >= 2) {
        if (rank == 0) {
            number = 42;
            MPI_Send(&number, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
            printf("Process 0 sent %d\n", number);
        }
        else if (rank == 1) {
            MPI_Recv(&number, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            printf("Process 1 received %d\n", number);
        }
    }

    int data;
    if (rank == 0) data = 100;

    MPI_Bcast(&data, 1, MPI_INT, 0, MPI_COMM_WORLD);
    printf("Process %d has data %d\n", rank, data);

    int local = rank + 1;
    int global_sum;

    MPI_Reduce(&local, &global_sum, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0)
        printf("Global Sum = %d\n", global_sum);

    MPI_Finalize();
    return 0;
}

Overwriting mpi_program.c


In [15]:
!mpicc mpi_program.c -o mpi_program

In [16]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./mpi_program

Hello from process 2 out of 4
Hello from process 1 out of 4
Hello from process 0 out of 4
Process 0 sent 42
Process 0 has data 100
Process 1 received 42
Process 1 has data 100
Hello from process 3 out of 4
Process 3 has data 100
Process 2 has data 100
Global Sum = 10
